# 三列小火车共用轨道，怎样全部到站？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{
  "cbs_status": "OPTIMAL_WITHIN_TIME_MODEL",
  "cbs_sum_cost": 17,
  "priority_sum_cost": 20,
  "cbs_makespan": 7,
  "cbs_high_level_nodes": 28,
  "collision_free": true
}
CBS方案总到达时间17，优先方案20；CBS高层展开28个节点。逐步合法性、同点冲突、对向交换和终点占用均检查通过。最优结论限定在本离散动作、总到达时间目标和30步时间窗模型内；实际列车的速度、制动和安全间隔未建模。


## 1. 补全：交换碰撞
两辆车在一条边上对向交换位置，不能只检查同点碰撞。

In [2]:
swap=[[(0,0),(0,1)],[(0,1),(0,0)]]
issue=first_conflict(swap)
assert issue[0]=="e" and issue[3]==1
print(issue)


('e', 0, 1, 1, (0, 0), (0, 1))


## 2. 修改：改变优先顺序
检查所有6种优先次序；失败应记录，不能删除失败次序再报告全成功。

In [3]:
results=[]
for order in itertools.permutations(range(3)):
    paths=prioritized(order)
    if paths is not None: assert validate_paths(paths)
    results.append((order,None if paths is None else sum(len(p)-1 for p in paths)))
assert len(results)==6
print(results)


[((0, 1, 2), 20), ((0, 2, 1), 17), ((1, 0, 2), 20), ((1, 2, 0), 17), ((2, 0, 1), 17), ((2, 1, 0), 17)]


## 3. 迁移：仓库机器人到站后停留
一车先到终点，另一车之后经过该点，应检测碰撞。再构造不冲突的等待方案。

In [4]:
bad=[[(0,0),(0,1)],[(1,2),(0,2),(0,1)]]
assert first_conflict(bad)[0]=="v"
good=[[(0,0),(0,1)],[(1,2),(1,1),(1,0)]]
assert first_conflict(good) is None
print("抵达后的占用必须继续纳入预约表。")


抵达后的占用必须继续纳入预约表。
